# 图片不多时，怎样利用已有模型完成分类？：参考答案

每道题先写预期结果再运行。练习中的 TODO 需要自行完成；能顺序运行不代表已完成练习。

In [1]:
from pathlib import Path
CASE_DIR = Path.cwd()
# 载入本例的可修改函数和数据，所有算法都在同目录 solution.py。
exec(compile((CASE_DIR / "solution.py").read_text(encoding="utf-8"), str(CASE_DIR / "solution.py"), "exec"))

{
  "raw_pixel_accuracy": 0.92,
  "frozen_resnet_accuracy": 0.87,
  "train_samples": 150,
  "test_samples": 100,
  "backbone_trainable_parameters": 0,
  "input_size": 112,
  "domain_warning": "ImageNet to digits; teaching workflow only",
  "elapsed_seconds": 6.526
}


## 1. 补全：冻结检查
确认骨干没有可训练参数，且 3 张图片得到 3×512 特征。

In [2]:
assert sum((p.numel() for p in backbone.parameters() if p.requires_grad)) == 0
assert extract_features(X[te[:3]]).shape == (3, 512)
print('骨干已冻结')

骨干已冻结


## 2. 修改：只训练一个神经分类头
冻结 train_features，训练 Linear(512,10) 15 步；检查骨干未获梯度而分类头获梯度。

In [3]:
torch.manual_seed(19)
new_head = nn.Linear(512, 10)
opt = torch.optim.Adam(new_head.parameters(), lr=0.01)
ft = torch.tensor(train_features)
labels = torch.tensor(y[tr], dtype=torch.long)
for _ in range(15):
    opt.zero_grad()
    loss = nn.CrossEntropyLoss()(new_head(ft), labels)
    loss.backward()
    opt.step()
assert new_head.weight.grad is not None
assert all((p.grad is None for p in backbone.parameters()))
print(float(loss))

0.27421659231185913


[USER_HOME]\AppData\Local\Temp\ipykernel_31612\3460900847.py:13: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:836.)
  print(float(loss))


## 3. 迁移：特征用于奇偶分类
不重新提取图像特征，只把新任务的标签改成奇偶。说明这样节省的是哪个环节。

In [4]:
parity = LogisticRegression(max_iter=500).fit(train_features, y[tr] % 2)
result = parity.predict(test_features)
assert set(np.unique(result)) <= {0, 1} and len(result) == len(te)
print('奇偶准确率', accuracy_score(y[te] % 2, result))

奇偶准确率 0.9
